In [1]:
# pip install numpy  (ROOT comes from your PyROOT install)
import numpy as np
import ROOT

# -----------------------------
# 1) Reproducibility ("root"/seed)
# -----------------------------
rng = np.random.default_rng(7)

# -----------------------------
# 2) Inputs
# -----------------------------
n = 40
x = np.linspace(0, 10, n)

# -----------------------------
# 3) Dataset A: non-correlated (i.i.d. noise)
# -----------------------------
sigma_noise = 1.0
y_uncorr = rng.normal(0, sigma_noise, size=n)

# -----------------------------
# 4) Dataset B: correlated (GP sample with SE/RBF kernel + small noise)
# -----------------------------
def rbf_kernel(x1, x2, sigma_f=1.5, ell=1.2):
    x1 = np.asarray(x1)[:, None]
    x2 = np.asarray(x2)[None, :]
    sqdist = (x1 - x2) ** 2
    return (sigma_f**2) * np.exp(-0.5 * sqdist / (ell**2))

sigma_f = 1.5
ell = 1.2
K = rbf_kernel(x, x, sigma_f=sigma_f, ell=ell)
K += 1e-8 * np.eye(n)  # jitter

y_corr = rng.multivariate_normal(mean=np.zeros(n), cov=K)
y_corr_noisy = y_corr + rng.normal(0, 0.2, size=n)

# -----------------------------
# 5) ROOT plots side-by-side
# -----------------------------
ROOT.gStyle.SetOptStat(0)

c = ROOT.TCanvas("c", "Correlated vs Non-correlated", 1100, 450)
c.Divide(2, 1)

# Helper to make a TGraph from numpy arrays
def make_graph(xarr, yarr, name):
    g = ROOT.TGraph(len(xarr), xarr.astype(np.float64), yarr.astype(np.float64))
    g.SetName(name)
    g.SetMarkerStyle(20)
    g.SetMarkerSize(1.0)
    return g

# Left: non-correlated
c.cd(1)
g1 = make_graph(x, y_uncorr, "g_uncorr")
g1.SetTitle("Dataset A: non-correlated (i.i.d. noise);x;y")
g1.Draw("AP")

# Right: correlated
c.cd(2)
g2 = make_graph(x, y_corr_noisy, "g_corr")
g2.SetTitle("Dataset B: correlated (GP sample + small noise);x;y")
g2.Draw("AP")

c.Update()

# Keep window open if running as a script
input("Press Enter to exit...")

/opt/homebrew/Cellar/root/6.38.00/lib/root/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /Users/chanayo/.pyenv/versions/3.14.0/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


''